## Fake News Classifier Using LSTM

Dataset: https://www.kaggle.com/c/fake-news/data#

In [1]:
import pandas as pd

In [3]:
df=pd.read_csv('train.csv')

In [4]:
df.head()

,id,title,author,text,label
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,1
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,0
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",1
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,1
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,1


In [5]:
df.shape

(20800, 5)

In [6]:
df.head()

,id,title,author,text,label
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,1
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,0
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",1
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,1
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,1


In [7]:
df.isnull().sum()

,0
id,0
title,558
author,1957
text,39
label,0


In [8]:
###Drop Nan Values -- The data is text so we cannot replace the values so its better to delete records
# Also we have 20k records so 2k data won't make a great deal to it
df=df.dropna()


In [10]:
# Also we cannot replace a author name or a title on our own
df.isnull().sum()

,0
id,0
title,0
author,0
text,0
label,0


In [ ]:
df.head()

,id,title,author,text,label
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,1
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,0
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",1
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,1
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,1


In [11]:
## Get the Independent Features

X = df.drop('label', axis = 1)

In [14]:
## Get the Dependent features
y = df['label']

In [15]:
X.shape

(18285, 4)

In [16]:
y.shape

(18285,)

In [17]:
import tensorflow as tf

In [18]:
tf.__version__

'2.17.1'

In [20]:
from tensorflow.keras.layers import Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense

In [19]:
### Vocabulary size
voc_size=5000

### Onehot Representation
One Hot encoding tells us in what index is the particular word from a dictionary(voc_size) of 5000 words

Eg 1: vocab size is 10
['the glass of milk']==> [1, 6, 3, 5]
1. the ==> 1000000000
2. glass ==> 0000010000
3. of ==> 0010000000
4. milk ==> 0000100000

Eg 2: ['the glass of milk']==> [180, 405, 264, 53]
here the vocab size is 500
so it says:
1.  when 180th index is 1 from 500 indices then it represents "the"
2. when 405th index is 1 from 500 indices then it represents "glass"



In [21]:
messages=X.copy()

In [22]:
messages['title'][1]

'FLYNN: Hillary Clinton, Big Woman on Campus - Breitbart'

In [23]:
messages

,id,title,author,text
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ..."
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...
...,...,...,...,...
20795,20795,Rapper T.I.: Trump a ’Poster Child For White S...,Jerome Hudson,Rapper T. I. unloaded on black celebrities who...
20796,20796,"N.F.L. Playoffs: Schedule, Matchups and Odds -...",Benjamin Hoffman,When the Green Bay Packers lost to the Washing...
20797,20797,Macy’s Is Said to Receive Takeover Approach by...,Michael J. de la Merced and Rachel Abrams,The Macy’s of today grew from the union of sev...
20798,20798,"NATO, Russia To Hold Parallel Exercises In Bal...",Alex Ansary,"NATO, Russia To Hold Parallel Exercises In Bal..."


In [29]:
messages.reset_index(inplace=True)

In [30]:
messages

,index,id,title,author,text
0,0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...
1,1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...
2,2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ..."
3,3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...
4,4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...
...,...,...,...,...,...
18280,20795,20795,Rapper T.I.: Trump a ’Poster Child For White S...,Jerome Hudson,Rapper T. I. unloaded on black celebrities who...
18281,20796,20796,"N.F.L. Playoffs: Schedule, Matchups and Odds -...",Benjamin Hoffman,When the Green Bay Packers lost to the Washing...
18282,20797,20797,Macy’s Is Said to Receive Takeover Approach by...,Michael J. de la Merced and Rachel Abrams,The Macy’s of today grew from the union of sev...
18283,20798,20798,"NATO, Russia To Hold Parallel Exercises In Bal...",Alex Ansary,"NATO, Russia To Hold Parallel Exercises In Bal..."


In [31]:
import nltk
import re
from nltk.corpus import stopwords

In [26]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

## Data Preprocessing:

###1. Stemming using Porter Stemmer: stemming converts working ==> work this just reduces the input size
###2. Removing stopwords: Removing words and special charcters like the, and, is, #, @, etc

In [32]:
### Dataset Preprocessing
from nltk.stem.porter import PorterStemmer ##stemming purpose
ps = PorterStemmer()
corpus = []
for i in range(0, len(messages)):
    review = re.sub('[^a-zA-Z]', ' ', messages['title'][i])
    review = review.lower()
    review = review.split()

    review = [ps.stem(word) for word in review if not word in stopwords.words('english')]
    review = ' '.join(review)
    corpus.append(review)

In [33]:
corpus

['hous dem aid even see comey letter jason chaffetz tweet',
 'flynn hillari clinton big woman campu breitbart',
 'truth might get fire',
 'civilian kill singl us airstrik identifi',
 'iranian woman jail fiction unpublish stori woman stone death adulteri',
 'jacki mason hollywood would love trump bomb north korea lack tran bathroom exclus video breitbart',
 'beno hamon win french socialist parti presidenti nomin new york time',
 'back channel plan ukrain russia courtesi trump associ new york time',
 'obama organ action partner soro link indivis disrupt trump agenda',
 'bbc comedi sketch real housew isi caus outrag',
 'russian research discov secret nazi militari base treasur hunter arctic photo',
 'us offici see link trump russia',
 'ye paid govern troll social media blog forum websit',
 'major leagu soccer argentin find home success new york time',
 'well fargo chief abruptli step new york time',
 'anonym donor pay million releas everyon arrest dakota access pipelin',
 'fbi close hilla

In [34]:
corpus[1]

'flynn hillari clinton big woman campu breitbart'

In [35]:
onehot_repr=[one_hot(words,voc_size)for words in corpus]
onehot_repr

[[4372, 4821, 349, 2090, 3879, 10, 4432, 132, 3823, 695],
 [2556, 2911, 2074, 3139, 4417, 4023, 499],
 [1561, 2859, 103, 89],
 [1464, 893, 1465, 4661, 3756, 3357],
 [4564, 4417, 290, 2118, 3121, 3679, 4417, 558, 1589, 4866],
 [2543,
  279,
  2423,
  3652,
  837,
  4367,
  2678,
  118,
  3692,
  4871,
  4254,
  3614,
  4321,
  1045,
  499],
 [3266, 58, 2509, 3932, 1035, 4729, 4225, 3946, 463, 2941, 2033],
 [3352, 1460, 707, 685, 104, 1272, 4367, 920, 463, 2941, 2033],
 [3635, 4775, 4084, 1304, 4886, 3528, 2342, 4175, 4367, 565],
 [115, 4330, 1207, 1519, 821, 52, 1571, 23],
 [1318, 1261, 3567, 1378, 3578, 1660, 3849, 2626, 3251, 1270, 4178],
 [4661, 2763, 3879, 3528, 4367, 104],
 [4320, 4561, 2329, 4366, 4418, 575, 3313, 2875, 4413],
 [2679, 1449, 2673, 981, 4694, 2232, 3884, 463, 2941, 2033],
 [2140, 2296, 507, 3167, 2083, 463, 2941, 2033],
 [2235, 1259, 2561, 1982, 4676, 4058, 847, 1014, 255, 342],
 [573, 4677, 2911],
 [2413, 3840, 247, 4943, 4367, 3173, 3230, 499],
 [678, 2254, 2074, 

In [ ]:
corpus[1]

'flynn hillari clinton big woman campu breitbart'

In [ ]:
onehot_repr[1]

[2861, 147, 1342, 1829, 62, 4605, 721]

### Embedding Representation

### Padding is done --> to make sure all the length of the sentence should be same

In [36]:
sent_length=20
embedded_docs=pad_sequences(onehot_repr,padding='post',maxlen=sent_length)
print(embedded_docs)

[[4372 4821  349 ...    0    0    0]
 [2556 2911 2074 ...    0    0    0]
 [1561 2859  103 ...    0    0    0]
 ...
 [3023 1810 3099 ...    0    0    0]
 [4445  104 1354 ...    0    0    0]
 [   8 4441 3787 ...    0    0    0]]


In [37]:
embedded_docs[1]

array([2556, 2911, 2074, 3139, 4417, 4023,  499,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0], dtype=int32)

In [38]:
embedded_docs[0]

array([4372, 4821,  349, 2090, 3879,   10, 4432,  132, 3823,  695,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0], dtype=int32)

#### LSTM + Embedding layer

In [63]:
## Creating model
embedding_vector_features=40  ##features representation/ no of dimensions
model=Sequential()
model.add(Embedding(voc_size,embedding_vector_features,input_length=sent_length))
model.add(LSTM(100))
model.add(Dense(1,activation='sigmoid'))
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])
print(model.summary())

/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)              │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_3 (LSTM)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


In [44]:
len(embedded_docs),y.shape

(18285, (18285,))

In [64]:
import numpy as np
X_final=np.array(embedded_docs)
y_final=np.array(y)

In [65]:
X_final.shape,y_final.shape

((18285, 20), (18285,))

In [66]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.33, random_state=42)

### Model Training

In [67]:
### Finally Training
model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=10,batch_size=64)

Epoch 1/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.7748 - loss: 0.4204 - val_accuracy: 0.9195 - val_loss: 0.1994
Epoch 2/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9513 - loss: 0.1350 - val_accuracy: 0.9196 - val_loss: 0.2005
Epoch 3/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9701 - loss: 0.0901 - val_accuracy: 0.9175 - val_loss: 0.2113
Epoch 4/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9792 - loss: 0.0663 - val_accuracy: 0.9138 - val_loss: 0.2861
Epoch 5/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9861 - loss: 0.0436 - val_accuracy: 0.9168 - val_loss: 0.2595
Epoch 6/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9893 - loss: 0.0369 - val_accuracy: 0.9137 - val_loss: 0.3721
Epoch 7/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.9919 - loss: 0.0318 - val_accuracy: 0.9100 - val_loss: 0.2926
Epoch 8/10
192/192 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9941 - loss: 0.0230 - val_accuracy: 0

### Adding Dropout

Why Dropout:
During training, a dropout layer randomly sets a fraction of input units to zero, based on a given probability. Dropout layers can improve classification and prediction accuracy, and are a cost-effective way to control model complexity.

In [56]:
from tensorflow.keras.layers import Dropout
## Creating model
embedding_vector_features=40
model=Sequential()
model.add(Embedding(voc_size,embedding_vector_features,input_length=sent_length))
model.add(Dropout(0.3))
model.add(LSTM(100))
model.add(Dropout(0.3))
model.add(Dense(1,activation='sigmoid'))
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])

/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


### Performance Metrics And Accuracy

In [68]:
y_pred=model.predict(X_test)

189/189 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [69]:
y_pred=np.where(y_pred > 0.6, 1,0) ##AUC ROC Curve

In [70]:
from sklearn.metrics import confusion_matrix

In [71]:
confusion_matrix(y_test,y_pred)

array([[3158,  261],
       [ 292, 2324]])

In [74]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.9083678541839271

In [73]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.92      0.92      0.92      3419
           1       0.90      0.89      0.89      2616

    accuracy                           0.91      6035
   macro avg       0.91      0.91      0.91      6035
weighted avg       0.91      0.91      0.91      6035

